In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. Load data
# ============================================================

df = pd.read_csv("dataset.csv")



ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# ============================================================
# 2. Columns to remove
# ============================================================

columns_to_remove = [
    "meta_loads",
    "meta_leads",
    "commission_sum_that_day",

    "avg_item_price_lag1",
    "avg_item_price_lag7_avg",

    "avg_order_food_value_lag1",
    "avg_order_food_value_lag7_avg",

    "meta_cpc_usd",
    "meta_cpm_usd",
    "meta_active_campaigns",

    "google_ads_cost_usd_estimated",
    "google_ads_avg_cpc_usd",

    "blogger_spend_afn",
    "total_ad_spend_afn",

    "cancelled_orders",
    "successful_orders",

    "kabul_order_count",
    "herat_order_count",

    "online_payment_orders",

    "average_order_item_count",
    "average_order_value",

    "delivery_share_lag7_avg",
    "delivery_share_lag1",

    "avg_delivery_fee_lag7_avg",
    "avg_delivery_fee_lag1",

    "avg_discount_amount_lag1",
    "avg_discount_amount_lag7_avg",

    "new_active_restaurants",
    "rating_sum_that_day",
    "featured_that_day",

    "new_available_dishes",
    "count_that_day",
    "event_bookings",
    "new_featured_dishes",

    "cumulative_featured_restaurants",
    "cumulative_commission_sum",
    "avg_commission_rate",

    "blogger_views_reach_lag7_avg",
    "blogger_views_reach_lag1",

    "spend_lag_1",

    "meta_impressions_lag1",
    "meta_impressions_lag7_avg",

    "meta_link_clicks_lag1",

    "meta_ctr_percent_lag7_avg",

    "meta_cpm_usd_lag1",
    "meta_cpc_usd_lag7_avg",

    "google_ads_ctr_percent_estimated",
    "google_ads_ctr_percent_estimated_lag1",
    "google_ads_ctr_percent_estimated_lag7_avg",
]


# Remove only columns that actually exist
columns_to_remove = [
    col for col in columns_to_remove
    if col in df.columns
]

df = df.drop(columns=columns_to_remove)


# ============================================================
# 3. Target
# ============================================================

target = "orders_total"

X = df.drop(columns=[target])
y = df[target]


# ============================================================
# 4. Convert date into useful features
# ============================================================

if "date" in X.columns:

    X["date"] = pd.to_datetime(
        X["date"],
        errors="coerce"
    )

    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["day_of_year"] = X["date"].dt.dayofyear

    # Remove original date because LinearRegression
    # cannot directly use datetime.
    X = X.drop(columns=["date"])


# ============================================================
# 5. Remove columns that are not useful as raw text
# ============================================================

text_columns_to_drop = [
    "meta_campaign_names",
    "meta_result_types",
    "blogger_names",
    "blogger_content_types",
    "blogger_effect_names",
    "movie_af_campaign_name",
    "google_source_week",
    "day_of_week",
]

text_columns_to_drop = [
    col for col in text_columns_to_drop
    if col in X.columns
]

X = X.drop(columns=text_columns_to_drop)


# ============================================================
# 6. Identify numerical and categorical features
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32", "bool"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()


# ============================================================
# 7. Preprocessing
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    ),
])


preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])


# ============================================================
# 8. Linear Regression model
# ============================================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])


# ============================================================
# 9. Train/Test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 10. Train
# ============================================================

model.fit(X_train, y_train)


# ============================================================
# 11. Predict
# ============================================================

y_pred = model.predict(X_test)


# Orders cannot be negative
y_pred = np.maximum(y_pred, 0)


# ============================================================
# 12. Evaluation
# ============================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)


print("Linear Regression Results")
print("=" * 40)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")


# ============================================================
# 13. Show actual vs predicted
# ============================================================

results = pd.DataFrame({
    "actual_orders": y_test.values,
    "predicted_orders": np.round(y_pred, 2),
})

print("\nPredictions:")
print(results.head(20))